# 05: Rating basis inventory (S_rated)

**Diagnostic only.** Nothing here changes a result, an Athena table or the site metadata table. The pipelines keep `ac_capacity_kw` (Solar Analytics) and `s_99` (OEM).

Every AS/NZS 4777.2 quantity is scaled by S_rated, the inverter's rated (maximum) apparent power in kVA: the required Q, the ±4% tolerance band, the 20% power threshold between **zone A: response required** and **zone C: low power, voltage exposed**, and the Figure 2.1 minimum. This notebook asks how good a stand-in `ac_capacity_kw` is:

1. What the metadata can tell (model coverage, unidentified models).
2. Per model: `ac_capacity_kw` and the apparent power each site actually delivered (`s_99`).
3. `ac_capacity_kw` against datasheet ratings (private file, outside the repository).
4. What remains unchecked.

**Private data.** The datasheet table names manufacturers, so it lives outside the repository (`%LOCALAPPDATA%\ciccada\ratings\inverter_datasheet_ratings.csv`, or `CICCADA_RATINGS_CSV`). Outputs are stripped on commit; never paste results into markdown cells.

In [ ]:
%matplotlib inline

import sys
from pathlib import Path

import pandas as pd

HERE = Path.cwd().resolve()
REPO_ROOT = next(p for p in (HERE, *HERE.parents) if (p / "bms_sa_review").is_dir())
for path in (REPO_ROOT, REPO_ROOT / "bms_sa_review", REPO_ROOT / "bms_sa_review" / "shared"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import importlib

from bms_sa_review.shared import rating_inventory as R

importlib.reload(R)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

## Parameters

In [ ]:
SOURCE = "local"        # "local": local store metadata (free, sites in the local extract)
                        # "athena": all PV sites from the site metadata table (metadata only, megabytes)
TOP_MODELS = 40         # models shown in the tables
DATASHEETS = None       # None: default private path (see above); or a path to the CSV

if SOURCE == "athena":
    from bms_sa_review.shared.aws_config import aq
    from bms_sa_review.shared.ciccada_config import SAI
    print(R.athena_metadata_sql())          # read the query first: metadata only
    meta = R.load_site_metadata("athena", aq=aq, database=SAI)
else:
    print("local metadata:", R.local_metadata_path())
    meta = R.load_site_metadata("local")
print(f"{len(meta):,} sites")

## 1. What the metadata can tell

In [ ]:
display(R.field_inventory(meta))

## 2. Per model: `ac_capacity_kw` and the apparent power actually delivered

`s_99` is the 99th percentile of the apparent power a site delivered. It is capped by the PV array, the weather and export limits, so a ratio below 1 does not prove the rating right; a ratio clearly above 1 shows the inverter delivers more than `ac_capacity_kw`.

In [ ]:
models = R.model_table(meta, top=TOP_MODELS)
display(models)

ax = meta["s99_over_ac"].clip(0, 1.5).plot.hist(bins=60, figsize=(8, 3.5))
ax.axvline(1.0, color="black", lw=0.8)
ax.axvline(1.03, color="red", ls="--", lw=0.8, label="3% above ac_capacity_kw")
ax.set(xlabel="s_99 / ac_capacity_kw (per site)", ylabel="sites")
ax.legend();

## 3. `ac_capacity_kw` against the datasheets

`case`: **matches max kVA** (within 2%), **below max kVA (kW, not kVA)** (the metadata holds the active-power rating and the inverter may deliver more apparent power), **above max kVA**, **no datasheet**. `gap_pct` = how much S_rated is understated by `ac_capacity_kw`.

In [ ]:
datasheets = R.load_datasheets(DATASHEETS)
print(f"{len(datasheets)} models in the datasheet file")

all_models = R.model_table(meta, top=10**6)
comparison = R.compare_with_datasheets(all_models, datasheets)
display(R.rating_gap_summary(comparison, n_sites_total=len(meta)))

checked = comparison[comparison["case"] != "no datasheet"].sort_values("n_sites", ascending=False)
display(checked[["manufacturer", "model", "n_sites", "ac_capacity_kw", "nominal_kw", "max_kva",
                 "gap_pct", "case", "s99_over_ac_median", "share_sites_s99_above_ac_3pct", "note"]])
R.plot_gap(comparison);

In [ ]:
# Does the observed apparent power agree? Sites delivering > 3% above ac_capacity_kw, by case.
per_site = meta.merge(comparison[["manufacturer", "model", "case"]], on=["manufacturer", "model"], how="left")
per_site["case"] = per_site["case"].fillna("unidentified model")
display(per_site.groupby("case")["s99_over_ac"].agg(
    sites="size", median="median", share_above_3pct=lambda x: (x > 1.03).mean()).round(3))

## 4. What remains unchecked (next datasheets to add, by sites covered)

In [ ]:
todo = comparison[comparison["case"] == "no datasheet"].sort_values("n_sites", ascending=False)
todo["cum_share_of_all_sites_pct"] = (100 * todo["n_sites"].cumsum() / len(meta)).round(1)
display(todo[["manufacturer", "model", "n_sites", "ac_capacity_kw", "s99_over_ac_median",
              "cum_share_of_all_sites_pct"]].head(TOP_MODELS))

## 5. Candidate rating (future option only)

Datasheet max kVA where the model is known, else `ac_capacity_kw`. **No pipeline reads this.**

In [ ]:
candidate = R.candidate_rating(meta, datasheets)
display(candidate["candidate_source"].value_counts().rename("sites").to_frame())
changed = candidate[candidate["candidate_s_rated_kva"] > candidate["ac_capacity_kw"] * 1.02]
print(f"sites whose candidate S_rated is > 2% above ac_capacity_kw: {len(changed):,} "
      f"({100 * len(changed) / len(candidate):.1f}%)")